In [128]:
%run T_symb.ipynb
%run helpers.ipynb
import copy
from itertools import combinations

In [186]:
class cochain_complex:
    # I'll think about how to deal with pickling in a bit here
    def __init__(self,T_symb_obj,pickled_ad_degs=[]):
        self.alg=T_symb_obj
        self.heis_dim=T_symb_obj.heis_dim
        T_symb_obj.cochain_complex=self
        self.ext_alg=ext_alg(T_symb_obj,pickled_ad_degs)
        T_symb_obj.ext_alg=self.ext_alg
        self.basis={}
        self.basis_strs={}
        self.bases_initialized=set()
    
    def cochain(self,coeff_dict,wght='UNKNOWN',deg='UNKNOWN'):
        return cochain(coeff_dict,self,wght,deg)
    
    def sort_tuple(self,cochain_tuple):
        '''cochain_tuple: a tuple of str_reps of T_symb_basis_elt objs, representing a cochain
           returns: a rearrangement of cochain_tuple, descending in degree,
                    but leaving the final element of cochain_tuple invariant'''
        ext_list=list(cochain_tuple[0:len(cochain_tuple)-1])
        ext_result=self.alg.sort_basis_tuple(ext_list)
        return((tuple(list(ext_result[0])+[cochain_tuple[len(cochain_tuple)-1]]),ext_result[1]))
    
    def deg(c):
        '''c: a cochain object from self
           returns: deg(c) if c has homogeneous deg, 
             'Nil' if c is the zero cochain,'UNKNOWN' otherwise'''
        if len(c.coeff_dict.keys())==0:
            return 'Nil' # The zero cochain
        deg=len(list(c.coeff_dict.keys())[0])-1
        for A in c.coeff_dict:
            if len(A)-1!=deg:
                return 'UNKNOWN'
        return deg
    
    def tuple_wght(rep):
        '''rep: a tuple of strings representing a basic cochain from self
           returns: the wght of the corresponding cochain'''
        result=0
        for A in rep[0:len(rep)-1]:
            result-=self.alg.basis[self.alg.basis_strs.index(A)].wght
        result+=self.alg.basis[self.alg.basis_strs.index(rep[len(rep)-1])].wght
        return result
    
    def wght(c):
        '''c: a cochain object
           returns: wght(c) if c has homogeneous wght, 'UNKNOWN' otherwise'''
        if len(c.coeff_dict.keys())==0:
            return 'Nil' # The zero cochain
        wght=self.cochain_tuple_wght(list(c.coeff_dict.keys())[0])
        for A in c.coeff_dict:
            if self.cochain_tuple_wght(A)!=wght:
                return 'UNKNOWN'
        return wght 
    
    def init_basis(self,deg):
        if deg in self.bases_initialized: return None
        deg_subsets=list(combinations(self.alg.basis_strs,deg))
        cochain_subsets=[A+(B,) for A in deg_subsets for B in self.alg.basis_strs]
        self.basis_strs[deg]=[tuple(A) for A in cochain_subsets]
        self.basis[deg]=[self.cochain({A:1}) for A in self.basis_strs[deg]]
        self.bases_initialized.add(deg)
    
#     def set_cochain_ad_dict(self,deg):
#         for deg in deg_list:
#             for A in self.alg.basis:
                
#                 # reset the dicts
#                 A.ext_ad_dicts[deg]={}
#                 A.cochain_ad_dicts={}
                
#         # To do  

In [158]:
class ext_alg:
    def __init__(self,T_symb_obj,pickled_ad_degs=[]):
        self.alg=T_symb_obj
        self.heis_dim=T_symb_obj.heis_dim
        T_symb_obj.ext_alg=self
        self.basis={}
        self.basis_strs={}
        bases_initialized=set()
    
    def elt(self,coeff_dict,wght='UNKNOWN',deg='UNKNOWN'):
        return ext_elt(coeff_dict,self,wght,deg)
    
    def deg(self,ext_elt):
        '''ext_elt: an ext_T_symb_elt object
           returns: deg(ext_elt) if ext_elt has homogeneneous deg,
             'Nil' if ext_elt is the zero wedge, 'UNKNOWN' otherwise'''
        if len(ext_elt.coeff_dict.keys())==0:
            return 'Nil'
        deg=len(list(ext_elt.coeff_dict.keys())[0])
        for A in ext_elt.coeff_dict:
            if len(A)!=deg:
                return 'UNKNOWN'
        return deg
    
    def wedge_tuples(self,tuple1,tuple2):
        '''tuple1,tuple2: tuples of T_symb_basis_elt objects
           returns: (wedge,sgn), where wedge is a tuple representing tuple1 wedge tuple2
                   and sgn is -1 or 1'''
        #check for repeats
        if len(set(tuple1).union(set(tuple2)))!=len(tuple1)+len(tuple2):
            return 'Nil'
        return self.alg.sort_basis_tuple(tuple1+tuple2)
    
    def tuple_wght(self,rep):
        '''rep: a tuple representing an ext_T_symb_elt
           returns: the wght of the corresponding exterior element'''
        result=0
        for A in rep:
            result-=self.alg.basis[self.alg.basis_strs.index(A)].wght
        return result
    
    def wght(self,ext_elt):
        '''ext_elt: an ext_T_symb_elt object
           returns: wght(ext_elt) if ext_elt has homogeneous wght, 'UNKNOWN' otherwise'''
        if len(ext_elt.coeff_dict.keys())==0:
            return 'Nil' # The zero cochain
        wght=self.tuple_wght(list(c.coeff_dict.keys())[0])
        for A in c.coeff_dict:
            if self.tuple_wght(A)!=wght:
                return 'UNKNOWN'
        return wght 
    
    def init_basis(self,deg):
        if deg in self.bases_initialized: return None
        deg_subsets=list(combinations(self.alg.basis_strs,deg))
        self.basis_strs[deg]=[tuple(A) for A in deg_subsets]
        self.basis[deg]=[self.elt({A:1}) for A in self.basis_strs[deg]]
        self.bases_initialized.add(deg)
        
#     def set_ext_ad_dict(self,deg):
#         '''sets the attribute ext_ad_dict for self including all '''
#         for deg in deg_list:
#             for A in self.alg.basis:
                
#                 # reset the dicts
#                 A.ext_ad_dicts[deg]={}
#                 A.cochain_ad_dicts={}
                
#                 #To do

In [120]:
class cochain:
    def __init__(self,coeff_dict,parent,wght='UNKNOWN',deg='UNKNOWN'):
        '''coeff_dict:  tuple of T_symb_basis_elts objs as keys, coeffs as values
           wght (optional): homog. wght, if known'''
        self.parent=parent
        self.deg=deg
        self.wght=wght
        self.heis_dim=parent.heis_dim 
        self.coeff_dict=remove_zeros({parent.sort_tuple(A)[0]:
                                      parent.sort_tuple(A)[1]*coeff_dict[A] for A in coeff_dict})
                
    def __eq__(self,other):
        if other==0:
            return self.coeff_dict=={}
        return self.coeff_dict==other.coeff_dict
        
    def __add__(self,other):
        # zero cochain case
        if other==0: return copy.copy(self)
        if self.deg=='Nil': return copy.copy(other)
        if other.deg=='Nil': return copy.copy(self)
        
        result=copy.copy(self)
        
        ## If one wght is UNKNOWN
        if self.wght==other.wght or other.wght=='UNKNOWN':
            result.wght=self.wght
        elif self.wght=='UNKNOWN':
            result.wght=other.wght
        else:
            result.wght='UNKNOWN'       
        
        result.coeff_dict=merge_coeff_dicts(self.coeff_dict,other.coeff_dict)
            
        return result
    
    def __radd__(self,other):
        return self+other
    
    def __neg__(self):
        return self.parent.cochain({A:-self.coeff_dict[A] for A in self.coeff_dict},
                       wght=self.wght,deg=self.deg)
        
    def __sub__(self,other):
        return self+(-other)
    
    def __mul__(self,k):
        kself=copy.copy(self)
        kself.coeff_dict={A:k*self.coeff_dict[A] for A in self.coeff_dict}
        return kself
    
    def __rmul__(self,k):
        return self*k
    
    def __str__(self):
        return self.parent.alg.str_from_coeff_dict(self.coeff_dict)
    
    def __repr__(self):
        return self.parent.alg.str_from_coeff_dict(self.coeff_dict)
    
    # I think these are unecessary now?
    def __getstate__(self):
        return self.__dict__
    
    def __setstate__(self,d):
        self.__dict__=d
        
    def apply_cochain_map(c,ext_elt):
        '''c: a cochain object
           ext_elt: an ext_T_symb_elt object
           returns: T_symb_elt object representing c(ext_elt) or None if c.deg!=ext_elt.deg'''
        result=T_symb_elt([0]*len(T_symb_basis),ext_elt.heis_dim)
        for c_basis_elt in c.coeff_dict:
            ext_basis_elt=c_basis_elt[0:len(c_basis_elt)-1]
            if ext_basis_elt in ext_elt.coeff_dict:
                coeff=ext_elt.coeff_dict[ext_basis_elt]*c.coeff_dict[c_basis_elt]
                new_symb_list=[0]*len(T_symb_basis)
                new_symb_list[T_symb_basis_strs.index(c_basis_elt[len(c_basis_elt)-1])]=coeff
                result+=T_symb_elt(new_symb_list,ext_elt.heis_dim)
        return result
    
    
#     # To do: think about this
#     def coboundary(self):
#         result=cochain({},self.parent)
#         for key in self.coeff_dict():
#             result=result+self.coeff_dict[key]*coboundary_dict[key]
#         return result

In [124]:
class ext_elt():
    
    def __init__(self,coeff_dict,parent,wght='UNKNOWN',deg='UNKNOWN'):
        if coeff_dict=={}: ## The zero wedge has wght and deg 'Nil'
            self.deg='Nil'
            self.wght='Nil'
        else:
            self.deg=deg   
            self.wght=wght
        self.parent=parent
        self.heis_dim=parent.heis_dim
        self.coeff_dict=remove_antisymm_zeros(remove_zeros({parent.alg.sort_basis_tuple(A)[0]:
                                      parent.alg.sort_basis_tuple(A)[1]*coeff_dict[A] for A in coeff_dict}))
    
    def __str__(self):
        return self.parent.alg.str_from_coeff_dict(self.coeff_dict)
    
    def __repr__(self):
        return self.parent.alg.str_from_coeff_dict(self.coeff_dict)
    
    def __eq__(self,other):
        if other==0:
            return self.coeff_dict=={}
        return self.coeff_dict==other.coeff_dict
    
    def __neg__(self):
        return ext_T_symb_elt({A:-self.coeff_dict[A] for A in self.coeff_dict},
                              self.heis_dim,wght=self.wght,deg=self.deg)
    
    def __add__(self,other):
        # To Do: add a check for if heis_dims match here (and in similar places)
        if self==0: return copy.copy(other)
        if other==0: return copy.copy(self)
        new_wght='UNKNOWN'
        if self.wght!='UNKNOWN' and other.wght!='UNKNOWN':
            new_wght=self.wght+other.wght
        new_deg='UNKNOWN'
        if self.deg!='UNKNOWN' and other.deg!='UNKNOWN':
            new_deg=self.deg+other.deg
        return(ext_T_symb_elt(merge_coeff_dicts(self.coeff_dict, other.coeff_dict),
                              self.heis_dim,wght=new_wght,deg=new_deg))
    
    def __radd__(self,other):
        return self+other
    
    def __sub__(self,other):
        return self+(-other)
    
    def __mul__(self,other):
        return(ext_T_symb_elt({A:other*self.coeff_dict[A] for A in self.coeff_dict},
                              self.heis_dim,wght=self.wght,deg=self.deg))
    
    def __rmul__(self,other):
        return self*other
    
    def wedge(self,other):
        '''other: an ext_T_symb_elt or cochain object
           returns: the wedge product of self and other as an ext_T_symb_elt
           Note: Functionality only for wedges of deg <4, since vec_rep is used'''
        
        if type(other)==ext_elt:
            result=ext_T_symb_elt({},self.parent)
            for A in self.coeff_dict:
                for B in other.coeff_dict:
                    new_wedge=self.parent.wedge_tuples(A,B)
                    if new_wedge!='Nil':
                        result+=ext_T_symb_elt({new_wedge[0]:new_wedge[1]*self.coeff_dict[A]*other.coeff_dict[B]},self.parent)
            return result
        
        if type(other)==cochain:
            result=cochain({},other.parent)
            for A in self.coeff_dict:
                for B in other.coeff_dict:
                    new_wedge=self.parent.wedge_tuples(A,B[0:len(B)-1])
                    new_cochain=(new_wedge[0]+B[len(B)-1:len(B)],new_wedge[1])
                    result+=cochain({new_cochain[0]:new_cochain[1]*self.coeff_dict[A]*other.coeff_dict[B]},
                                    self.parent.alg.cochain_complex)
            return result
        
        # To do: Make sure this is what I want this to do...
        if type(other)==T_symb_elt:
            result=cochain({},self.alg.cochain_complex)
            for ext_tuple in self.coeff_dict:
                for i in range(len(T_symb_basis)):
                    A=T_symb_basis_strs[i]
                    coeff=self.coeff_dict[ext_tuple]*other.vec_rep[i]
                    result+=cochain({ext_tuple+(A,):coeff},self.alg.cochain_complex)
            return result
        
        if type(other)==T_symb_basis_elt:
            vec=[0]*len(T_symb_basis)
            vec[T_symb_basis.index(other)]=1
            return self.wedge(T_symb_elt(vec,self.heis_dim))
    
    # I think these are unecessary now?
    def __getstate__(self):
        return self.__dict__
    
    def __setstate__(self,d):
        self.__dict__=d
        

In [187]:
test_alg=T_symb(7)
test_complex=cochain_complex(test_alg)
test_ext=ext_alg(test_alg)
test_cochain1=test_complex.cochain({('X','Y'):1})
test_cochain2=test_complex.cochain({('Y','e1'):-2})
test_cochain3=test_complex.cochain({('Y','e1'):2,('X','Y'):-1})
test_e1=test_ext.elt({('Y',):1})
test_e2=test_ext.elt({('X',):2})

In [91]:
-test_cochain-2*test_cochain2==test_cochain3

True

# Coboundary Operator

In [ ]:
# # To do: refactor this
# # No need to run this if coboundary_dict has been pickled

# coboundary_dict={}
# # keys: tuples representing elementary cochains of deg 0,1,2
# # values: cochain objects representing coboundary(key)

# time0=time.time()
# for deg in [0,1,2]:
#     for c in cochain_basis[deg]:
#         c_tuple=cochain_basis_tuples[deg][cochain_basis[deg].index(c)]
#         dc=cochain({},heis_dim)
#         for ext_elt in ext_basis[deg+1]:
#             dc_ext_elt=T_symb_elt([0]*len(T_symb_basis),heis_dim)

#             # First, compute im:=dc(ext_elt), a degree zero cochain
#             for i in range(len(ext_elt)):
#                 # (-1)**i*[ai,c(a0,...\hat ai,...ak)]
#                 no_i=ext_T_symb_elt({tuple(ext_elt[0:i]
#                                            +ext_elt[i+1:len(ext_elt)]):(-1)**i},heis_dim)
#                 im=apply_cochain_map(c,no_i)
#                 dc_ext_elt+=ad(globals()[ext_elt[i]],im)
#             for i in range(len(ext_elt)):
#                 for j in range(i+1,len(ext_elt)):
#                     # (-1)**(i+j)*c([ai,aj],a0,...,\hat ai,...\hat aj,...ak)
#                     ext_elt2=ext_T_symb_elt({ext_elt[0:i]+ext_elt[i+1:j]
#                                              +ext_elt[j+1:len(ext_elt)]:1},heis_dim)
#                     T_elt1=ad(T_symb_basis_dict[ext_elt[i]],T_symb_basis_dict[ext_elt[j]])
#                     ext_elt1=ext_T_symb_elt({(T_symb_basis_strs[i],):T_elt1.vec_rep[i] 
#                                              for i in range(len(T_symb_basis))},heis_dim)
#                     # wedging with the zero cochain gives zero, so treat deg=1 separately
#                     if ext_elt2==0:
#                         new_wedge=ext_elt1
#                     else:
#                         new_wedge=ext_elt1.wedge(ext_elt2)
#                     dc_ext_elt+=apply_cochain_map((-1)**(i+j)*c,new_wedge)
#             dc+=ext_T_symb_elt({ext_elt:1},heis_dim).wedge(dc_ext_elt)
#         coboundary_dict[c_tuple]=dc
# time1=time.time()        
# print('coboundary_dict set\ntotal time: '+hrs_min_sec(time1-time0))